# SENATI | Módulo: Machine Learning y Deep Learning
## CASO EVALUADO · Estimar el costo de una impresión 3D
**Instructor: Ing. Gunar Castro Iñigo**  
**Duración sugerida: 20 minutos**  
**Modalidad:** taller guiado en Jupyter Notebook  

**Estimados estudiantes:** Hoy vamos a aprender construyendo. Ejecuten cada celda en orden, lean los comentarios y expliquen los resultados con sus propias palabras. No se trata solamente de obtener un número: necesitamos comprender qué significa.

**Propósito:** desarrollar sin copiar la solución un proyecto de regresión con datos de impresión 3D

> **Aviso de calidad de datos:** Los archivos CSV del curso son **datos sintéticos generados para enseñanza**, con patrones plausibles, no registros oficiales ni mediciones reales. No deben emplearse para tomar decisiones reales sobre personas, equipos o empresas.

**Trabajo desarrollado por:** COCA CCORPA LORENA ABIGAIL  
**Institución:** SENATI · 2026  
**Actividad:** Caso práctico evaluado — estimar costo de impresión 3D  
**Nota:** Los datos suministrados por el taller son simulados; los resultados y pruebas son cálculos realizados sobre esos CSV.

# Encargo evaluado | Centro de fabricación digital

**Contexto simulado:** El área de impresión 3D registra características de trabajos de producción: impresora, peso en gramos, horas, material, calidad y costo total en soles. Tu trabajo consiste en entrenar un modelo que estime el costo de un nuevo pedido.

**Entrega:** notebook ejecutado + CSV con predicciones de prueba + respuesta interpretativa en el cuaderno.

### Requerimientos (orden obligatorio)

1. Importar `evaluacion_costos_impresion3d.csv`; verificar filas, tipos y nulos.
2. Calcular media, mediana, varianza muestral y desviación estándar del costo.
3. Elaborar dos observaciones a partir de agrupar por `material` y `calidad`.
4. Construir `X` con `impresora`, `peso_gramos`, `horas_impresion`, `material`, `calidad`; `y` será `costo_total_soles`.
5. Dividir en entrenamiento (80 %) y prueba (20 %), `random_state=42`.
6. Construir un Pipeline con OneHotEncoder para las tres columnas categóricas, y un regressor de Scikit-learn.
7. Informar MAE, R² y MAE de referencia (media entrenamiento). **No usar datos de prueba para entrenar**.
8. Predecir un nuevo pedido: Modelo B, 110 gramos, 4.5 horas, PLA, calidad Alta.
9. Exportar a CSV las columnas de X de prueba y agregar `real`, `estimado`.
10. Redactar una conclusión humana: qué mide MAE, qué puede salir mal con datos reales y qué falta validar.

**Se calificará** código funcional (30 %), preparación y evaluación rigurosa (30 %), interpretación (25 %) y entrega/documentación (15 %).

> Los valores del dataset son sintéticos y el precio observado es simulado; el sistema no define tarifas comerciales reales.

<div style="background-color:#FFF0A8;color:#463600;border-left:7px solid #D6A400;padding:14px 18px;border-radius:9px;margin:10px 0 16px"><strong>🟨 INICIO DEL DESARROLLO DE LA ACTIVIDAD</strong><br>Desde aquí se realiza los diez requerimientos del caso de impresión 3D. Se conserva el enunciado original y el código se ejecuta en orden.</div>

### Paso inicial: abrir los datos

El código siguiente ya funciona. Completa el resto del ejercicio creando celdas nuevas debajo.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def buscar_dataset(nombre):
    opciones=[Path.cwd()/nombre,Path.cwd()/'03_CASO_PRACTICO_EVALUADO'/nombre,
              Path.cwd().parent/'03_CASO_PRACTICO_EVALUADO'/nombre,
              Path.cwd()/'02_DATASETS_CSV'/nombre,
              Path.cwd().parent/'02_DATASETS_CSV'/nombre]
    for op in opciones:
        if op.exists():return op
    raise FileNotFoundError(nombre)

datos=pd.read_csv(buscar_dataset('evaluacion_costos_impresion3d.csv'))
print('Datos cargados:',datos.shape)
print(datos.head().to_string(index=False))
# TU TURNO: escribe nuevas celdas para ejecutar los requisitos 1-10.

Datos cargados: (700, 7)
registro_id impresora  peso_gramos  horas_impresion material  calidad  costo_total_soles
   IMP-0001  Modelo C          254              9.3      PLA Estandar              91.48
   IMP-0002  Modelo A          170             14.2      PLA Estandar              86.20
   IMP-0003  Modelo A          285             12.3      PLA Estandar             107.38
   IMP-0004  Modelo B          160             14.4      ABS Estandar              91.07
   IMP-0005  Modelo A           77             13.9      PLA     Alta              60.60


In [2]:
# Estadística descriptiva y comparaciones por categorías
print('Dimensiones:', datos.shape)
print('Tipos de datos:', datos.dtypes.to_dict())
print('Ausentes:', datos.isna().sum().to_dict())
costos = datos['costo_total_soles']
print('Media:',round(costos.mean(),3),'Mediana:',round(costos.median(),3),'Varianza muestral:',round(costos.var(ddof=1),3),'Desviación muestral:',round(costos.std(ddof=1),3))
print('Por material:')
print(datos.groupby('material')['costo_total_soles'].agg(['count','mean','median']).round(2))
print('Por calidad:')
print(datos.groupby('calidad')['costo_total_soles'].agg(['count','mean','median']).round(2))

Dimensiones: (700, 7)
Tipos de datos: {'registro_id': dtype('O'), 'impresora': dtype('O'), 'peso_gramos': dtype('int64'), 'horas_impresion': dtype('float64'), 'material': dtype('O'), 'calidad': dtype('O'), 'costo_total_soles': dtype('float64')}
Ausentes: {'registro_id': 0, 'impresora': 0, 'peso_gramos': 0, 'horas_impresion': 0, 'material': 0, 'calidad': 0, 'costo_total_soles': 0}
Media: 83.231 Mediana: 83.815 Varianza muestral: 1014.929 Desviación muestral: 31.858
Por material:
          count   mean  median
material                      
ABS         230  82.06   81.21
PETG        225  90.68   89.00
PLA         245  77.49   80.12
Por calidad:
          count   mean  median
calidad                       
Alta        356  86.86   87.93
Estandar    344  79.47   81.08


**Dos observaciones por comprobar:** (1) Los costos promedio difieren según el material usado; (2) los promedios varían según calidad. Son asociaciones descriptivas del conjunto simulado, no evidencia de relaciones causales.

### Respuestas cortas

1. **Por material:** PETG tiene el promedio mayor (S/ 90.68) y PLA el menor (S/ 77.49).
2. **Por calidad:** calidad Alta promedia S/ 86.86 y Estándar S/ 79.47. Son datos simulados; no prueban causalidad.

In [3]:
# Pipeline completo sin fuga: codificación de categorías + regresión
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
columnas_numericas = ['peso_gramos','horas_impresion']
columnas_categoria = ['impresora','material','calidad']
columnas = ['impresora','peso_gramos','horas_impresion','material','calidad']
X = datos[columnas]; y = datos['costo_total_soles']
Xent,Xpru,yent,ypru=train_test_split(X,y,test_size=0.20,random_state=42)
preparador=ColumnTransformer([('categorias',OneHotEncoder(handle_unknown='ignore'),columnas_categoria)],remainder='passthrough')
modelo=Pipeline([('preparador',preparador),('regresor',Ridge(alpha=1.0))])
modelo.fit(Xent,yent)
estimadas=modelo.predict(Xpru)
mae=mean_absolute_error(ypru,estimadas);r2=r2_score(ypru,estimadas)
mae_referencia=mean_absolute_error(ypru,np.full(len(ypru),yent.mean()))
print(f'Entrenamiento: {len(Xent)}; prueba: {len(Xpru)}')
print(f'MAE modelo {mae:.3f} S/; R² {r2:.3f}; MAE media entrenamiento {mae_referencia:.3f} S/')
nuevo_pedido=pd.DataFrame([{'impresora':'Modelo B','peso_gramos':110,'horas_impresion':4.5,'material':'PLA','calidad':'Alta'}])[columnas]
print('Nuevo pedido S/:',round(float(modelo.predict(nuevo_pedido)[0]),2))
salida=Xpru.copy()
salida['real']=ypru.to_numpy();salida['estimado']=estimadas
salida.to_csv('predicciones_prueba.csv',index=False)
print('CSV exportado: predicciones_prueba.csv')

Entrenamiento: 560; prueba: 140
MAE modelo 4.203 S/; R² 0.973; MAE media entrenamiento 27.331 S/
Nuevo pedido S/: 48.78
CSV exportado: predicciones_prueba.csv


**Decisión del modelo:** Ridge reduce la sensibilidad a coeficientes muy grandes, y OneHotEncoder evita tratar material, impresora y calidad como números ordinales. **MAE** mide el error promedio absoluto en soles. **R²** compara contra la referencia constante. **Limitaciones:** no se incluyen mermas, mantenimiento, energía real ni cambios de precios. Para uso real se necesitan pedidos históricos auténticos, muestras nuevas, validación y revisión de costes.

### Evidencias de trabajo

Al final agrega un párrafo con estos puntos:

- **Modelo elegido y justificación:** Ridge con OneHotEncoder; procesa categorías sin tratarlas como números y controla coeficientes.
- **MAE y R² obtenidos:** MAE **S/ 4.203** y R² **0.973**.
- **Comparación con la media:** referencia MAE **S/ 27.331**; el modelo obtuvo menor error.
- **Conclusión y limitaciones:** la prueba nueva estima **S/ 48.78**; validar con pedidos reales, energía, material y mantenimiento.

**Importante:** El desarrollo se muestra en este mismo cuaderno, después del enunciado.